## **SETUP**

In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [2]:
data= pd.read_csv("C:/Users/visco/RoadtoML/harga pangan/Tabel Harga Ayam Berdasarkan Daerah.csv", na_values='-')
data.head()

,No,Komoditas (Rp),03/ 01/ 2022,04/ 01/ 2022,05/ 01/ 2022,06/ 01/ 2022,07/ 01/ 2022,10/ 01/ 2022,11/ 01/ 2022,12/ 01/ 2022,...,18/ 05/ 2026,19/ 05/ 2026,20/ 05/ 2026,21/ 05/ 2026,22/ 05/ 2026,25/ 05/ 2026,26/ 05/ 2026,27/ 05/ 2026,28/ 05/ 2026,29/ 05/ 2026
0,I,Daging Ayam,"35,350","35,300","35,050","34,800","34,950","36,400","37,000","37,250",...,"35,850","35,750","35,500","35,600","35,850","35,850","35,750",NaN,NaN,"35,750"


## **DATA PROCESSING**

1. Konversi format harga (hilangin koma)

In [3]:
numeric_cols = data.columns[2:]
data[numeric_cols] = data[numeric_cols].replace(',', '', regex=True)

for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')

2. Konversi format wide ke long

In [4]:
data_long = data.melt(id_vars=[data.columns[1]], 
                  var_name='Tanggal', 
                  value_name='Harga')

data_long.columns = ['Komoditas', 'Tanggal', 'Harga']

3. Konversi format tanggal ke datetime

In [5]:
data_long['Tanggal'] = pd.to_datetime(data_long['Tanggal'], errors='coerce')
data_long = data_long.dropna(subset=['Tanggal'])
data_long.set_index('Tanggal', inplace=True)

C:\Users\visco\AppData\Local\Temp\ipykernel_31684\251046819.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data_long['Tanggal'] = pd.to_datetime(data_long['Tanggal'], errors='coerce')


In [6]:
data_long.head()

,Komoditas,Harga
Tanggal,,
2022-03-01,Daging Ayam,35350
2022-04-01,Daging Ayam,35300
2022-05-01,Daging Ayam,35050
2022-06-01,Daging Ayam,34800
2022-07-01,Daging Ayam,34950


### **HANDLE MISSING VALUE**

In [7]:
missing_cols = data_long.columns[data_long.isnull().any()]

print("Kolom dengan missing value:")
print(missing_cols)

Kolom dengan missing value:
Index(['Harga'], dtype='object')


In [8]:
data_long['Harga'] = data_long['Harga'].fillna(
    data_long['Harga'].rolling(window=7, min_periods=1).mean()
)

C:\Users\visco\AppData\Local\Temp\ipykernel_31684\3932372653.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_long['Harga'] = data_long['Harga'].fillna(


Tambah 3 kolom Harga dari 3 hari sebelumnya

In [9]:
data_long['h-1'] = data_long['Harga'].shift(1)
data_long['h-2'] = data_long['Harga'].shift(2)
data_long['h-3'] = data_long['Harga'].shift(3)

Hapus baris yang gapunya data masa lalu/masa depan

In [10]:
data_model = data_long.dropna()

In [11]:
missing_cols = data_model.columns[data_model.isnull().any()]

print("Kolom dengan missing value:")
print(missing_cols)

Kolom dengan missing value:
Index([], dtype='object')


In [12]:
data_model.head()

,Komoditas,Harga,h-1,h-2,h-3
Tanggal,,,,,
2022-06-01,Daging Ayam,34800.0,35050.0,35300.0,35350.0
2022-07-01,Daging Ayam,34950.0,34800.0,35050.0,35300.0
2022-10-01,Daging Ayam,36400.0,34950.0,34800.0,35050.0
2022-11-01,Daging Ayam,37000.0,36400.0,34950.0,34800.0
2022-12-01,Daging Ayam,37250.0,37000.0,36400.0,34950.0
